%md
## **Before starting the lab, you need a Unity Catalog volume to store the file**. Follow these steps to create one:

- In the Databricks workspace sidebar, click Catalog.
- In the Catalog Explorer,right side click on + icon and create a Catalog ( DP750_Lab_02 ), type =standard & default storage.
- Then Create expand Catalog and generate a new schema (MySchema) by clicking on + icon and create Volume.
- Click the ⋮ menu next to default, then select Create > Volume. (Bronze)
- Enter lab_data as the volume name, leave the type as Managed, and click Create.

## Exercise 3: Install Libraries Notebook-Scoped

At HealthBridge Analytics, different teams sometimes need different versions of the same library. Rather than creating a new cluster for every variation, you can install libraries **notebook-scoped** — they apply only to the current notebook session and do not affect other users or notebooks on the same compute resource.

In this exercise, you install the `faker` library notebook-scoped and verify it works correctly.

[faker](https://faker.readthedocs.io/) is a Python library that generates realistic fake data — such as names, addresses, phone numbers, dates, and more — for use in testing, prototyping, and seeding databases. It supports many locales and data categories, making it easy to produce large volumes of plausible synthetic data without using real personal information.

### Task 3.1 — Install the `faker` library notebook-scoped

Your team relies on `faker` to generate synthetic patient records for pipeline testing. Install it as a notebook-scoped library so the package is available to this notebook only and doesn't affect other users sharing the same compute.

**Hint:** Use a `%pip` magic command to install the package. Pinning an exact version (e.g., `faker==40.8.0`) is a good practice for reproducibility.

### Code Explanation

```python
pip install faker==40.8.0
```

- `pip` → Python package manager.
- `install` → Command to install a package.
- `faker` → Library used to generate fake data (names, emails, addresses, etc.).
- `==` → Specifies an exact package version.
- `40.8.0` → Version of the Faker library to install.

**Purpose:** Installs version **40.8.0** of the Faker library in the current Python environment.

In [0]:
pip install faker==40.8.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Task 3.2 — Verify the installation

Before using `faker` in a real pipeline, confirm it was installed correctly by:

1. Importing the `Faker` class from the `faker` library.
2. Creating a `Faker` instance.
3. Printing a randomly generated **full name** and **date of birth**.


### Code Explanation

```python
from faker import Faker

faker = Faker()
print(f"Full Name: {faker.name()}")
print(f"Date of Birth: {faker.date_of_birth()}")
```

- `from faker import Faker` → Imports the `Faker` class from the Faker library.
- `faker = Faker()` → Creates an instance of the Faker class to generate fake data.
- `faker.name()` → Generates a random full name.
- `faker.date_of_birth()` → Generates a random date of birth.
- `print()` → Displays the generated values on the screen.
- `f"..."` → An f-string used to embed values inside a string.

**Sample Output**
```text
Full Name: John Smith
Date of Birth: 1995-08-12
```

In [0]:
from faker import Faker

faker = Faker()
print(f"Full Name: {faker.name()}")
print(f"Date of Birth: {faker.date_of_birth()}")

Full Name: Kelly Thomas
Date of Birth: 1978-04-30


## Exercise 4: Generate and Analyze Synthetic Patient Data

Now that `faker` is installed, put it to work. Your team needs a synthetic dataset of patient admission records to test a new data pipeline. The data must look realistic enough to validate transformations and aggregations, but must contain **no real patient information** to comply with HIPAA policies.

In this exercise, you generate a synthetic dataset and run a basic analysis on it.

### Task 4.1 — Generate a synthetic patient admissions DataFrame

Use `faker` and the Spark session (`spark`) to create a **Spark DataFrame** with **100 synthetic patient admission records**. Each record must include:

| Column | Description |
|---|---|
| `patient_id` | Integer, 1 to 100 |
| `full_name` | Randomly generated full name |
| `date_of_birth` | Random date between `1940-01-01` and `2005-12-31`, as a string |
| `admission_date` | Random date between `2023-01-01` and `2025-12-31`, as a string |
| `diagnosis_code` | One of: `I21`, `J18`, `E11`, `K80`, `N39` (randomly chosen) |

Display the first 10 rows of the resulting DataFrame.



### Code Explanation

```python
from faker import Faker
from datetime import datetime
import random
```
- `Faker` → Generates fake data.
- `datetime` → Used to define date ranges.
- `random` → Used to select random values.

```python
faker = Faker()
```
- Creates a Faker object for generating fake records.

```python
diagnosis_codes = ['I21', 'J18', 'E11', 'K80', 'N39']
```
- Stores a list of diagnosis codes.

```python
data = [
```
- Starts creating a list of patient records.

```python
{
    "patient_id": i + 1,
```
- Assigns a unique patient ID from 1 to 100.

```python
"full_name": faker.name(),
```
- Generates a random patient name.

```python
"date_of_birth": faker.date_between(...).strftime("%Y-%m-%d"),
```
- Generates a random birth date between 1940 and 2005.
- `strftime()` formats the date as `YYYY-MM-DD`.

```python
"admission_date": faker.date_between(...).strftime("%Y-%m-%d"),
```
- Generates a random admission date between 2023 and 2025.

```python
"diagnosis_code": random.choice(diagnosis_codes)
```
- Randomly selects a diagnosis code from the list.

```python
for i in range(100)
```
- Repeats the process 100 times to create 100 records.

```python
df = spark.createDataFrame(data)
```
- Converts the Python list into a Spark DataFrame.

```python
display(df.limit(10))
```
- Shows the first 10 records of the DataFrame.

### Purpose
Generate **100 fake patient records** and display the first **10 rows** in Databricks.

In [0]:
from faker import Faker
from datetime import datetime
import random

faker = Faker()
diagnosis_codes = ['I21', 'J18', 'E11', 'K80', 'N39']

data = [
    {
        "patient_id": i + 1,
        "full_name": faker.name(),
        "date_of_birth": faker.date_between(start_date=datetime(1940, 1, 1), end_date=datetime(2005, 12, 31)).strftime("%Y-%m-%d"),
        "admission_date": faker.date_between(start_date=datetime(2023, 1, 1), end_date=datetime(2025, 12, 31)).strftime("%Y-%m-%d"),
        "diagnosis_code": random.choice(diagnosis_codes)
    }
    for i in range(100)
]

df = spark.createDataFrame(data)
display(df.limit(10))

admission_date,date_of_birth,diagnosis_code,full_name,patient_id
2025-03-26,1964-03-06,J18,Cassandra White,1
2025-02-21,1951-04-14,E11,Mrs. Diane Dillon DDS,2
2023-02-07,2002-01-24,K80,Sandra Ramirez,3
2024-08-26,1991-01-03,J18,Terry Conley,4
2023-09-24,1967-06-08,N39,Mrs. Connie Harris,5
2023-09-28,1980-12-11,K80,Cody Jordan,6
2024-02-10,1961-05-15,N39,Kathleen Jordan,7
2023-11-18,1997-10-28,N39,Julie Torres,8
2023-06-06,1989-02-09,K80,James Lara,9
2025-12-14,1941-06-27,J18,Mrs. Cathy Carter,10


### Task 4.2 — Analyze admissions by diagnosis code

The clinical informatics team wants to know which diagnosis codes appear most frequently in the admission data. Using the Spark DataFrame you created in Task 4.1:

1. Count the number of admissions per `diagnosis_code`.
2. Order the results from **most to least** admissions.
3. Display the result.

### Code Explanation

```python
from pyspark.sql.functions import desc
```
- `desc` → Imports a function to sort data in descending order.

```python
display(df.groupBy("diagnosis_code").count().orderBy(desc("count")))
```
- `groupBy("diagnosis_code")` → Groups records by diagnosis code.
- `.count()` → Counts the number of records in each group.
- `orderBy(desc("count"))` → Sorts the groups by count in descending order.
- `display()` → Displays the result in a table or chart in Databricks.

### Purpose
Count the number of patients for each diagnosis code and display them from highest to lowest count.

In [0]:
from pyspark.sql.functions import desc

display(df.groupBy("diagnosis_code").count().orderBy(desc("count")))

diagnosis_code,count
N39,31
E11,23
K80,17
I21,15
J18,14


In [0]:
# TODO: Count admissions per diagnosis_code, ordered from most to least